# 01 · Joint person-search training (ResNet-50 baseline / Swin-T warm start)

Trains one Faster R-CNN (backbone + FPN + RPN + detection head) **jointly** with a ReID head. Detection uses the standard RPN and RoI-head losses on the full frames. ReID uses the supervised NT-Xent loss (SupCon) or a batch-hard triplet loss on **ground-truth person crops**. The crops are re-encoded by the same backbone and positives must come from different frames. It is the architecture of the *ResNet baseline* system in `main.ipynb` (diagram 1).

The notebook has two configurations (`CONFIG` below). Run it once per configuration:

| `CONFIG` | Backbone | ReID loss | Output file | Used by | Run time (T4) |
|---|---|---|---|---|---|
| `"resnet_baseline"` | ResNet-50 (ImageNet) | SupCon, τ = 0.1 | `resnet_baseline.pt` | `main.ipynb` (ResNet baseline) | ≈ 3.1 h |
| `"joint_swin_t_triplet"` | Swin-T (ImageNet) | triplet, margin 0.3 | `joint_swin_t_triplet.pt` | warm start of `02_swin_detector`, validation detector of `04_dinov3_reid` | ≈ 2.6 h |

Shared settings: frames resized to 256×384, P×K = 8×3 identity batches, AdamW (lr 1e-4, weight decay 1e-4), 50 epochs, AMP. 10% of the training frames (seed 0) are held out for validation. The checkpoint with the lowest validation loss is kept.

### How to run
1. Create a Kaggle notebook from this file (*File → Import Notebook*), then attach the inputs listed below with *Add Input*.
2. Settings: **Accelerator → GPU T4 ×1**. Internet is only needed if you set `USE_WANDB = True`.
3. **Run All.** The inputs are found automatically wherever Kaggle mounts them under `/kaggle/input`.

Weights & Biases logging is **off** by default (`USE_WANDB = False`), so no account or secret is needed. To log curves and upload checkpoints, set `USE_WANDB = True` and store your key as the Kaggle secret / environment variable `WANDB_API_KEY`.

**Inputs**

| Input | Kaggle source | Used for |
|---|---|---|
| PRW dataset | `edoardomerli/prw-person-re-identification-in-the-wild` | frames, annotations, queries |
| Code | `saadqasimabbasi/prw-person-search-code` | the project modules in `src/` (and the DINOv3 code snapshot) |

## 1. Setup: locate the Kaggle inputs
This cell scans `/kaggle/input` once and finds the project code (the `prw-person-search-code` dataset), the PRW dataset and the checkpoint files this notebook needs. It fails with a clear message if one is missing, then adds the code folder to `sys.path`. Outputs go to `/kaggle/working`. Set the environment variables `PRW_INPUT_ROOT` / `PRW_WORKING` to run outside Kaggle.

> **Run time:** a few seconds.

In [ ]:
import os
import sys
from pathlib import Path

import torch

INPUT_ROOT = Path(os.environ.get("PRW_INPUT_ROOT", "/kaggle/input"))
WORKING = Path(os.environ.get("PRW_WORKING", "/kaggle/working"))
USE_WANDB = False  # True: log curves and upload checkpoints to W&B (needs WANDB_API_KEY)
os.environ["WANDB_MODE"] = "online" if USE_WANDB else "disabled"
os.environ["WANDB_DIR"] = str(WORKING)

PROJECT_MODULES = {"model.py", "training.py", "prw_dataset.py", "prw_evaluation.py"}
REQUIRED_FILES = {}
SKIPPED_FOLDERS = {"frames", "annotations", "query_box", ".git"}  # thousands of images, nothing to find


def scan_inputs():
    """Walk the input folder once and return the code, DINOv3, PRW and checkpoint locations."""
    code_dir = dinov3_dir = prw_root = None
    found = {}
    for directory, subfolders, files in os.walk(INPUT_ROOT, followlinks=True):
        directory = Path(directory)
        if prw_root is None and "query_info.txt" in files and "frames" in subfolders:
            prw_root = directory
        subfolders[:] = [name for name in subfolders if name not in SKIPPED_FOLDERS]
        if code_dir is None and PROJECT_MODULES.issubset(files):
            code_dir = directory
        if dinov3_dir is None and "hubconf.py" in files and directory.name.startswith("dinov3"):
            dinov3_dir = directory
        for name in REQUIRED_FILES.intersection(files):
            found.setdefault(name, directory / name)
    return code_dir, dinov3_dir, prw_root, found


CODE_DIR, DINOV3_DIR, PRW_ROOT, INPUT_FILES = scan_inputs()
missing = [label for label, value in [("code dataset (project .py files)", CODE_DIR),
                                      ("PRW dataset", PRW_ROOT)] if value is None]
missing += [f"input file {name}" for name in sorted(REQUIRED_FILES - set(INPUT_FILES))]
if missing:
    raise FileNotFoundError("Missing inputs (see 'How to run'): " + ", ".join(missing))
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
WORKING.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Code:  ", CODE_DIR)
print("PRW:   ", PRW_ROOT)
for name, path in sorted(INPUT_FILES.items()):
    print("Input: ", name, "->", path)
print("Device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "(no GPU: enable one in the settings)")

## 2. Configuration
Pick the configuration, then everything else follows from it. The model and run settings are the ones stored in the published checkpoints. `RESUME_FROM` can point to a `latest.pt` from an interrupted session (attach it as an input).

> **Run time:** < 1 s.

In [ ]:
import random

CONFIG = "resnet_baseline"  # or "joint_swin_t_triplet"
CONFIGS = {
    "resnet_baseline": dict(backbone="resnet50", reid_loss="supcon"),
    "joint_swin_t_triplet": dict(backbone="swin_t", reid_loss="triplet"),
}
BACKBONE, REID_LOSS = CONFIGS[CONFIG]["backbone"], CONFIGS[CONFIG]["reid_loss"]

SEED = 0
IMAGE_SIZE = (256, 384)
VAL_FRACTION = 0.10
P, K = 8, 3                # identities per batch, frames per identity
VAL_P, VAL_K = 4, 2        # smaller fixed validation batches; every selected ID has a positive
NUM_WORKERS = 2
NUM_EPOCHS = 50
VAL_MAX_BATCHES = 20
TRAIN_LR = 1e-4
WEIGHT_DECAY = 1e-4
ADAMW_BETAS = (0.9, 0.999)
ADAMW_EPS = 1e-8
REID_TEMPERATURE = 0.1     # SupCon
TRIPLET_MARGIN = 0.3       # triplet
RESUME_FROM = None         # e.g. Path("/kaggle/input/<previous-output>/latest.pt")
RUN_NAME = f"prw-exp04-{BACKBONE}-p8-k3-crop-to-crop-{REID_LOSS}"

torch.manual_seed(SEED)
random.seed(SEED)
print(CONFIG, "->", BACKBONE, REID_LOSS)

## 3. Data: train/validation split and P×K identity batches
The official PRW training frames are split 90/10 with a fixed seed. `PKIdentityBatchSampler` builds batches of P identities × K frames, so every identity in a batch has a positive from another frame. The validation batches are fixed (seeded), so validation losses are comparable across epochs.

> **Run time:** ≈ 10 s.

In [ ]:
from torch.utils.data import DataLoader

from prw_dataset import PRWPersonSearchDataset, PRWSubset, PKIdentityBatchSampler, prw_collate_fn

full_train = PRWPersonSearchDataset(PRW_ROOT, split="train", image_size=IMAGE_SIZE)
indices = torch.randperm(len(full_train), generator=torch.Generator().manual_seed(SEED)).tolist()
val_count = round(len(indices) * VAL_FRACTION)
val_dataset = PRWSubset(full_train, indices[:val_count])
train_dataset = PRWSubset(full_train, indices[val_count:])

train_sampler = PKIdentityBatchSampler(train_dataset, p=P, k=K)
val_sampler = PKIdentityBatchSampler(val_dataset, p=VAL_P, k=VAL_K, seed=SEED)
train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, collate_fn=prw_collate_fn,
                          num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
val_loader = DataLoader(val_dataset, batch_sampler=val_sampler, collate_fn=prw_collate_fn,
                        num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
print(f"train frames: {len(train_dataset)}, validation frames: {len(val_dataset)}")
print(f"P×K batches per epoch: {len(train_sampler)}, nominal batch size: {P * K}")
print(f"fixed validation batches: {len(val_sampler)}")

## 4. Model and run configuration
`PersonSearchModel` is the Faster R-CNN with the ReID head (`src/model.py`). The ImageNet backbone weights come from torchvision. `run_config` is stored inside every checkpoint, so `main.ipynb` can rebuild the model exactly.

> **Run time:** ≈ 10 s (downloads the torchvision ImageNet weights on first use).

In [ ]:
from model import PersonSearchModel

model_config = dict(
    backbone_name=BACKBONE, pretrained=True, metric_loss=REID_LOSS,
    triplet_margin=TRIPLET_MARGIN, reid_temperature=REID_TEMPERATURE,
    anchor_sizes=((16,), (32,), (64,), (128,)),
    anchor_ratios=((2.0, 3.0, 4.0),) * 4,
    rpn_fg_iou_thresh=0.7, rpn_bg_iou_thresh=0.3,
    roi_fg_iou_thresh=0.7, roi_bg_iou_thresh=0.3,
    rpn_pre_nms_top_n_train=1000, rpn_post_nms_top_n_train=512,
    rpn_pre_nms_top_n_test=500, rpn_post_nms_top_n_test=100,
)
run_config = {
    **model_config, "dataset": "PRW", "train_split": "official train (90%)",
    "validation_split": "official train (10%)", "test_split": "official test",
    "image_size": IMAGE_SIZE, "seed": SEED, "P": P, "K": K,
    "val_P": VAL_P, "val_K": VAL_K, "val_sampling_seed": SEED,
    "nominal_batch_size": P * K, "epochs": NUM_EPOCHS,
    "reid_training": "gt_crop_to_crop_cross_image_exp04",
    "query_crop_size": (128, 64), "gallery_reid": "standalone_detected_crops",
    "optimizer": "AdamW", "lr": TRAIN_LR, "weight_decay": WEIGHT_DECAY,
    "adamw_betas": ADAMW_BETAS, "adamw_eps": ADAMW_EPS,
    "pretrained_weights": f"torchvision {BACKBONE} ImageNet-1K",
    "amp": device.type == "cuda",
}
model = PersonSearchModel(**model_config).to(device)
print(f"trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.1f} M")

## 5. Train, validate and checkpoint
Each epoch trains on all P×K batches (detection + ReID losses), then computes the same losses on the fixed validation batches. `latest.pt` is overwritten every epoch, and `best.pt` keeps the epoch with the lowest total validation loss. At the end, the best checkpoint is restored and copied to `/kaggle/working/<CONFIG>.pt`. That file is the one published in the weights dataset.

> **Run time (Kaggle T4):** ≈ 3 h for ResNet-50, ≈ 2.5 h for Swin-T.

In [ ]:
import shutil

import wandb

from training import make_grad_scaler, start_run, train_one_epoch, validate

run = start_run(run_config, name=RUN_NAME)
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                              lr=TRAIN_LR, weight_decay=WEIGHT_DECAY, betas=ADAMW_BETAS, eps=ADAMW_EPS)
scaler = make_grad_scaler(device)
checkpoint_dir = WORKING / "checkpoints" / CONFIG
checkpoint_dir.mkdir(parents=True, exist_ok=True)

start_epoch, global_step, best_val_total = 0, 0, float("inf")
if RESUME_FROM is not None:
    checkpoint = torch.load(RESUME_FROM, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])
    start_epoch, global_step = checkpoint["epoch"], checkpoint["step"]
    best_val_total = checkpoint.get("best_val_total", float("inf"))
    print(f"resuming after epoch {start_epoch} from {RESUME_FROM}")

for epoch in range(start_epoch, NUM_EPOCHS):
    history = train_one_epoch(model, train_loader, optimizer, device, run=run, scaler=scaler,
                              start_step=global_step, epoch=epoch, log_every=20)
    global_step += len(history.get("total", []))
    val_metrics = validate(model, val_loader, device, run=run, step=max(global_step - 1, 0),
                           max_batches=VAL_MAX_BATCHES, epoch=epoch + 1)
    improved = val_metrics["total"] < best_val_total
    best_val_total = min(best_val_total, val_metrics["total"])
    checkpoint = {
        "epoch": epoch + 1, "step": global_step, "best_val_total": best_val_total,
        "model_config": model_config, "run_config": run_config,
        "model_state_dict": model.state_dict(), "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(), "val_metrics": val_metrics,
    }
    torch.save(checkpoint, checkpoint_dir / "latest.pt")
    if improved:
        torch.save(checkpoint, checkpoint_dir / "best.pt")
    print(f"epoch {epoch + 1}/{NUM_EPOCHS}: val_total={val_metrics['total']:.4f}, best={best_val_total:.4f}", flush=True)

best_checkpoint = torch.load(checkpoint_dir / "best.pt", map_location=device, weights_only=False)
model.load_state_dict(best_checkpoint["model_state_dict"])
shutil.copy2(checkpoint_dir / "best.pt", WORKING / f"{CONFIG}.pt")
if USE_WANDB:
    artifact = wandb.Artifact(f"{CONFIG}-best", type="model", metadata=run_config)
    artifact.add_file(str(WORKING / f"{CONFIG}.pt"))
    run.log_artifact(artifact)
run.finish()
print(f"training complete: best epoch {best_checkpoint['epoch']} saved as {WORKING / f'{CONFIG}.pt'}")

## 6. Test evaluation with the official `eval_search_prw`
The best checkpoint is evaluated once on the official PRW test split: all 2,057 queries against detected boxes (confidence ≥ 0.5) in the 6,112 gallery frames. Gallery detections are cropped from the original frames and re-encoded, exactly as in `main.ipynb`. Test data is never used for training or checkpoint selection.

> **Run time (Kaggle T4):** ≈ 15 min.

In [ ]:
from prw_evaluation import evaluate_prw

model.eval()
result = evaluate_prw(model, PRW_ROOT, device, score_threshold=0.5, iou_threshold=0.5,
                      detector_image_size=IMAGE_SIZE, gallery_embedding_mode="crop", official_eval=True,
                      batch_size=2, query_batch_size=32, gallery_crop_batch_size=32)
print(f"{CONFIG}: official mAP = {result['official_mAP']:.2%}, top-1 = {result['official_top1']:.2%}")